In [1]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import boston_housing
from sklearn import datasets
from tensorflow.keras.utils import to_categorical

# -1. Matrices, Gradients, Hessians

Given the function $f(x1, x2) = (1 - x1)^2 + 5 (x2 - x1^2)^2$, compute:

### a) Gradient $\nabla f(1, 1)$

$ ∂f/∂x = -2(1-x1)-20x1(x1-x2^2) $ <br>
$ ∂f/∂y = 10(x2-x1)^2 $

$∇f(1, 1) = [0; 0]$

### b) Hessian $\nabla^2 f(1, 1)$


$ ∂^2f/∂x^2 = 60x1^2-20x2 + 2 $ <br>
$ ∂^2f/∂x1x2 = -20x1 $ <br>
$ ∂^2f/∂x2^2 = 10 $

$∇^2 f(x, y) = [42\ -20; -20\ 10] $

### c) Is the hessian positive definite? Why? $p^T H p > 0$

Yes, <br>
$ H11 = 42 > 0 $ <br>
$ det(H) = 20 > 0 $


# 0. Implementing 'Package' Supporting Functions

In [2]:
def relu(x):
  """Computes relu of function."""
  return x * (x >= 0)


def sigmoid(x):
  """Computes sigmoid of function."""
  return 1 / (1 + np.exp(-x))

def linear(x):
  """Return linear function of itself."""
  return x

def d_linear(x):
    return np.ones_like(x)

def d_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

def d_relu(x):
    return (x >= 0).astype(np.float32)

activations = {
    "linear": linear,
    "relu": relu,
    "sigmoid": sigmoid
}

d_activations = {
    "linear": d_linear,
    "relu": d_relu,
    "sigmoid": d_sigmoid
}

In [3]:
def initialize_weights(shapes, outputs):
  """Initializes weights of model according to shape.
     Args:
       shapes = [784, 300, 10]
       outputs = ["linear", "sigmoid"]

     returns:
       model with uniform random weights [-1,+1], zero bias and output function
       [
        [random(784, 300), zeros(300), "linear"]
        [random(300, 10), zeros(10), "sigmoid"]
       ]

  """
  model = []

  for i in range(len(outputs)):
      in_dim = shapes[i]
      out_dim = shapes[i + 1]

      # weights ~ Uniform[-1, 1]
      W = np.random.uniform(-1, 1, size=(in_dim, out_dim))
      b = np.zeros(out_dim)

      model.append([W, b, outputs[i]])

  return model


model = initialize_weights([4, 10, 3], ['sigmoid', 'sigmoid'])
#[[w1, b1, 'sigmoid'], [w2, b2, 'sigmoid']]

In [4]:
def forward(x, model):
    """Performs forward pass of training step.

     Args:
       x: input tensor of shape (B, N0)
       model: list of model weights (see initialize weights)
     Returns:
       List containing dictionary { "y": y, "z": z } for each layer of network.
       [{"y": y1, "z": z1}, {"y": y2, "z": z2}]
    """
    result = []
    y = x
    for layer in model:
      # layer = [ w[i], b[i], 'relu']
      z = np.dot(y, layer[0]) + layer[1]  # (B,N1) + (N1,N2)
      y = activations[layer[2]](z)
      result.append({'y': y, 'z': z})

    return result

In [5]:
def predict(x, model):
  """Predicts the output of a model.

     Args:
       x: input tensor of shape (B, Ni)
       model: list of model weights (see initialize weights)
     Returns:
       Prediction of model, with the same shape as the labeled data (B, No).
  """
  fwd = forward(x, model)
  return fwd[-1]["y"]

In [6]:
def backward(y, x, model, loss):
  """Computes backward step of training.
     Args:
       y: labeled data of size (B, No)
       x: input tensor of shape (B, Ni)
       model: list of model weights (see initialize weights)
       loss: one of ("mse", "binary_crossentropy")
     Returns:
       tuple with loss evaluation of (y, predict(x)) and list of dictionary
       containing { "dw": dw, "db": db } for each layer of network. Remember
       that shape of dw for each layer should be equal to shape of weight for
       the same layer.
       [{"dw": dw1, "db": db1}, {"dw": dw2, "db": db2}]
  """
  # forward pass
  cache = forward(x, model)
  p = cache[-1]["y"]  # prediction

  # loss function
  if isinstance(loss, str):
    if loss == "mse":
      loss_fn = mse
    elif loss == "binary_crossentropy":
      loss_fn = binary_crossentropy
    else:
      raise ValueError("loss must be 'mse' or 'binary_crossentropy'")
  else:
    loss_fn = loss

  loss_val = loss_fn(y, p)

  B = x.shape[0]

  # gradient
  if loss == "mse":
    No = y.shape[1]
    dL_dp = (2.0 / (B * No)) * (p - y)
  elif loss == "binary_crossentropy":
    # BCE is mean over batch of sum over outputs
    eps = 1e-7
    p_clip = np.clip(p, eps, 1.0 - eps)
    dL_dp = (1.0 / B) * ((p_clip - y) / (p_clip * (1.0 - p_clip)))
  else:
    raise ValueError("Loss Function")

  # --- backprop through layers ---
  grads = [None] * len(model)

  # last layer activation derivative
  zL = cache[-1]["z"]
  layer = model[-1][2]
  delta = dL_dp * d_activations[layer](zL)  # (B, No)

  for i in reversed(range(len(model))):
    W, b, act = model[i]

    a_prev = x if i == 0 else cache[i - 1]["y"]  # (B, Ni)
    dw = a_prev.T @ delta                        # (Ni, No) matches W shape
    db = np.sum(delta, axis=0)                   # (No,)

    grads[i] = {"dw": dw, "db": db}

    if i > 0:
      z_prev = cache[i - 1]["z"]
      act_prev = model[i - 1][2]
      delta = (delta @ W.T) * d_activations[act_prev](z_prev)

  return loss_val, grads

In [7]:
def update(weights, dweights, alpha):
  """Gradient descent for weights and biases."""
  for i in range(len(weights)):
    weights[i][0] += - alpha * dweights[i]["dw"]
    weights[i][1] += - alpha * dweights[i]["db"]

In [8]:
def mse(y, p):
  """Computes Mean-Square Error between y and p.
     Args:
       y: labeled data of size (B, No)
       p: predicted label of size (B, No)
     Returns:
       MSE of y-p
  """
  assert p.shape == y.shape
  return np.mean((y - p)**2)

def binary_crossentropy(y, p):
  """Computes binary crossentropy between y and p.
     Args:
       y: labeled data of size (B, No)
       p: predicted label of size (B, No)
     Returns:
       BCE of (y, p) = mean(sum(y log(p) + (1-y) log(1-p)))
  """
  assert p.shape == y.shape, f"Prediction shape {p.shape} != label shape {y.shape}"
  eps = 1e-7
  p = np.clip(p, eps, 1.0 - eps)
  return -np.mean(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p), axis=1))

# 1. Creating and Training Linear Regression on Boston Dataset

In [9]:
def load_data():
  (x_train, y_train), (x_test, y_test) = boston_housing.load_data()

  # condition data to be in a format you need to use

  return (x_train, y_train), (x_test, y_test)

In [10]:
def train_network():

  (x_train, y_train), (x_test, y_test) = load_data()

  # linear network
  # plot training and test loss over time in jupyter notebook
  y_train = y_train.reshape(-1, 1)
  y_test  = y_test.reshape(-1, 1)

  shapes = [x_train.shape[1], 1]

  print(shapes)

  outputs = ["linear"]
  model = initialize_weights(shapes, outputs)

  fwd_results = forward(x_train, model)

  # what's the alpha you should use?
  alpha = 0.000002

  for i in range(300):
    loss, dweights = backward(y_train, x_train, model, "mse")
    update(model, dweights, alpha)
    print(i, loss)

train_network()

[13, 1]
0 32908.92625971496
1 22872.27528748297
2 18546.736471584652
3 15229.273307974847
4 12531.349711125995
5 10328.079704868813
6 8528.189397373548
7 7057.728362814037
8 5856.330582622722
9 4874.6936907553145
10 4072.5498533507907
11 3417.0101322992923
12 2881.212329318712
13 2443.2166118731784
14 2085.103511128286
15 1792.2372078760288
16 1552.6638181122737
17 1356.619940166356
18 1196.1312584325444
19 1064.6847012224928
20 956.9606742657043
21 868.6143612566041
22 796.0971001252021
23 736.5104913291752
24 687.4872401667096
25 647.0938342169985
26 613.7510547161546
27 586.1690538773877
28 563.294329009473
29 544.2664133969336
30 528.3825033875868
31 515.0685674119555
32 503.8557491487
33 494.36109470683385
34 486.27181146743743
35 479.33241142362493
36 473.3342104469981
37 468.1067517673027
38 463.51080106158275
39 459.43262516223007
40 455.7793191663471
41 452.4749898314422
42 449.4576383467143
43 446.6766143223925
44 444.09053632390777
45 441.6655934585899
46 439.37415818867055


- Change training function to collect training and test loss

# 2. Classification on Iris Dataset

In [11]:
import sklearn

def load_iris():
  iris = datasets.load_iris()
  x = iris.data.astype(np.float32)
  y = iris.target

  y = to_categorical(y, np.max(y)+1).astype(np.float32)

  x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(
        x, y,
        test_size=0.2,
        random_state=1,
        stratify=np.argmax(y, axis=1)  # stratify by class labels)
  )

  # (B,) -> {0, 1, 2}
  # (B, 3) -> 0 -> y[:, 0] = 1, 1 -> y[:, 1] = 1

  # need to do conditioning on the dataset

  return (x_train, y_train), (x_test, y_test)

In [12]:
def accuracy(y, p): # (B, 3)
    # y = (1, 0, 0), p = (0.2, 0.5, 0.3)
    # y = 0, p = 1
    return np.mean(np.argmax(y, axis=-1) == np.argmax(p, axis=-1))

def train_network():
  (x_train, y_train), (x_test, y_test) = load_iris()

  # plot training and test loss over time in jupyter notebook
  # plot training and test accuracy over time in jupyter notebook

  outputs = ["relu", "relu", "sigmoid"]
  shapes = [x_train.shape[1], 30, 10, y_train.shape[1]]

  print(shapes)

  # which alpha should you use?

  alpha = 0.005
  model = initialize_weights(shapes, outputs)

  for i in range(100):
    loss, dweights = backward(y_train, x_train, model, "binary_crossentropy")
    update(model, dweights, alpha)
    print(i, loss)

train_network()

[4, 30, 10, 3]
0 6.779330676320518
1 4.918532116950181
2 3.4696307894970944
3 2.7091942259313098
4 2.4566714431378
5 2.365056667931918
6 2.3189230860131245
7 2.2879708407725485
8 2.262858842612777
9 2.2406122255162253
10 2.220229933750183
11 2.201395627480872
12 2.1840665410704196
13 2.1678286772085245
14 2.152568251221543
15 2.138190883609854
16 2.1246548915901005
17 2.1119039097821335
18 2.099975224218541
19 2.08860656467193
20 2.0777488342263792
21 2.0673242271454186
22 2.057276102771475
23 2.047698375153715
24 2.0382179230229998
25 2.0288095027471322
26 2.0195891281074787
27 2.010490263787108
28 2.001601811649406
29 1.992239195834851
30 1.982244477963275
31 1.9715261782216194
32 1.960558913583985
33 1.9492591652079094
34 1.9380417740798521
35 1.926323283894638
36 1.9149076256905677
37 1.9035663668047371
38 1.8919236563475466
39 1.8803878115627704
40 1.8689773694195158
41 1.8578664158437903
42 1.848001435685434
43 1.8392991383369934
44 1.8326241882830294
45 1.8270688705245992
46 1.8

# 3. Packages These Days Use Automatic Differentiation Like AutoGrad

In [13]:
import autograd.numpy as np
from autograd import elementwise_grad as grad

ModuleNotFoundError: No module named 'autograd'

In [ ]:
x = np.arange(6).astype(np.float32)
x

In [ ]:
def f(x):
    return 3.0 * np.power(x, 2) + 5.0 * x - 4.0

In [ ]:
grad_f = grad(f)

In [ ]:
grad_f(x)

In [ ]:
(f(x + 0.01) - f(x - 0.01)) / 0.02

Comments:

1. I said in the class to always check the shapes to see the operations. dw and w have transposed shapes.
2. If you look at the notes, you will see that y's and z's have the same shape
3. If you need to debug on the fly, put the following line in the code. You can search on the web but basically, you can do step by step computation with pdb.

```python
import pdb; pdb.set_trace()
```